# `torch.compile` Speed-ups

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

`torch.compile` traces your model into an FX graph, optimises it (operator fusion, kernel selection), and compiles to Triton/AOT kernels. Often gives a free 1.3–2× speed-up on a modern GPU with zero code changes.


## Mathematical Formulation

Conceptually $f_{\text{compiled}} \equiv f$, but evaluated by a fused graph instead of eager-mode dispatch. The first call pays a compilation cost; subsequent calls reuse the cached graph.


## Implementation


In [ ]:
import time, torch
import torch.nn as nn


In [ ]:
def benchmark(model, x, warmup=5, runs=20):
    if x.is_cuda: torch.cuda.synchronize()
    for _ in range(warmup): _ = model(x)
    if x.is_cuda: torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs): _ = model(x)
    if x.is_cuda: torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mlp = nn.Sequential(nn.Linear(1024, 1024), nn.GELU(), nn.Linear(1024, 1024)).to(device).eval()
x = torch.randn(64, 1024, device=device)


## Experiment


In [ ]:
eager_t = benchmark(mlp, x)
mlp_c = torch.compile(mlp)
compiled_t = benchmark(mlp_c, x)
print(f'eager     : {eager_t*1e3:.2f} ms')
print(f'compiled  : {compiled_t*1e3:.2f} ms')
print(f'speed-up  : {eager_t / max(compiled_t, 1e-9):.2f}x')


## Discussion

- The first call after `torch.compile` takes seconds (tracing + compiling). Plan for it during warmup.
- Use `mode='reduce-overhead'` for inference-heavy workloads, `'max-autotune'` when you can afford long compile times.
- Beware of graph breaks (Python-side branching, in-place ops with side effects) — they kill perf.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
